```
# Lab type: prompt
# Course: ML402 — Reinforcement Learning
# Lesson: Actor-Critic Methods and PPO
# Task: You are given three AI-generated PPO configurations for a real scenario.
#       For each one: diagnose what is wrong, fix the config, and rewrite the
#       prompt that would have produced a correct config in the first place.
```

## Scenario

You are training a PPO agent to control a simulated robotic arm that must pick up an object and place it in a target zone. The environment has the following properties:

- **Observation space**: 24-dimensional continuous vector (joint positions, velocities, object pose)
- **Action space**: 6-dimensional continuous (joint torques)
- **Reward structure**: sparse — the agent receives `+1.0` only when the object lands in the target zone; all other steps give `0.0`
- **Typical episode length**: 200–400 steps before success; 500 steps max (then truncated)
- **Training budget**: 5 000 000 timesteps, 8 parallel environments

Your teammate asked an AI tool to generate a starting PPO config using `stable-baselines3`. The tool produced three variants. Two contain serious problems. Your job is to identify which configs are broken, explain why, and write corrected versions.

In [ ]:
# Install
!pip install gymnasium stable-baselines3

In [ ]:
# This cell is for inspection only — do not run a full training job.
# Treat the configs below as code to audit, not code to execute.

from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env

# Placeholder environment; substitute RoboticArm-v1 in a real run
ENV_NAME = "Pendulum-v1"


## Config A

```python
model_A = PPO(
    "MlpPolicy",
    make_vec_env(ENV_NAME, n_envs=8),
    n_steps=64,
    batch_size=512,
    n_epochs=10,
    learning_rate=3e-4,
    clip_range=0.2,
    ent_coef=0.01,
    verbose=1,
)
model_A.learn(total_timesteps=5_000_000)
```

### Diagnosis

**What is wrong with Config A?**

*(Write your answer here. Hint: think about how `n_steps` interacts with episode length and reward sparsity.)*

### Fix

In the cell below, write the corrected config with `n_steps` set to an appropriate value and a brief comment explaining your choice.

<details>
<summary>🔑 Reveal answer — Config A diagnosis</summary>

**The problem:** `n_steps=64` means each environment collects only 64 steps before an update — far shorter than the average episode length of 200–400 steps. With 8 envs this gives 512 transitions per rollout, and at most one-fifth of an episode per environment. In a sparse-reward environment the agent only earns `+1.0` when it completes a full successful episode, so the vast majority of 64-step rollouts contain zero reward. The advantage estimates for every transition in those rollouts are zero, the policy gradient is zero, and no learning can happen.

**The fix:** Set `n_steps` to at least 512 (to exceed average episode length per env) and preferably 1024 (to comfortably cover the 500-step maximum). This ensures each rollout spans multiple complete episodes across the 8 environments, giving the advantage estimator a realistic chance of seeing non-zero returns.

```python
model_A_fixed = PPO(
    "MlpPolicy",
    make_vec_env(ENV_NAME, n_envs=8),
    n_steps=1024,      # covers ~3–5 complete episodes per env; rollout sees reward signal
    batch_size=512,
    n_epochs=10,
    learning_rate=3e-4,
    clip_range=0.2,
    ent_coef=0.01,
    verbose=1,
)
```

</details>

In [ ]:
# Fix for Config A

# model_A_fixed = PPO(
#     "MlpPolicy",
#     make_vec_env(ENV_NAME, n_envs=8),
#     n_steps=???,          # your value here — justify in a comment
#     batch_size=512,
#     n_epochs=10,
#     learning_rate=3e-4,
#     clip_range=0.2,
#     ent_coef=0.01,
#     verbose=1,
# )


### Revised prompt for Config A

Below is the prompt your teammate used. Rewrite it so a fresh AI tool would generate a config with an appropriate `n_steps`.

**Original prompt:**
> "Write a Stable-Baselines3 PPO training script for a robotic arm task with 8 parallel environments and a 5M timestep budget."

**Your revised prompt:**

*(Write here. Be specific about episode length and reward structure in the prompt — an AI tool can only make correct hyperparameter choices when you give it this information.)*

<details>
<summary>🔑 Model prompt — Config A</summary>

**Example strong prompt:**

> Write a Stable-Baselines3 PPO config for a sparse-reward robotic pick-and-place task. The environment has a 24-dimensional continuous observation, a 6-dimensional continuous action space, and episodes of 200–400 steps (500-step hard limit). The reward is +1.0 only on task completion; all other steps give 0.0. Use 8 parallel environments and a 5 000 000 timestep budget. Set `n_steps` so that each rollout spans at least one full episode per environment (i.e., n_steps ≥ 500). Keep other hyperparameters at PPO defaults unless you have a specific reason to change them.

**Why it's strong:** It gives the model the episode length and reward structure it needs to reason about rollout coverage, and states the `n_steps` constraint explicitly as a floor rather than leaving it to a default that may be tuned for dense-reward environments.

</details>

---

## Config B

```python
model_B = PPO(
    "MlpPolicy",
    make_vec_env(ENV_NAME, n_envs=8),
    n_steps=1024,
    batch_size=512,
    n_epochs=10,
    learning_rate=3e-4,
    clip_range=0.2,
    ent_coef=0.0,
    verbose=1,
)
model_B.learn(total_timesteps=5_000_000)
```

### Diagnosis

**What is wrong with Config B?**

*(Write your answer here. Hint: consider what happens to a policy that never finds the reward in a sparse environment, and what keeps it from collapsing.)*

### Fix

In the cell below, write the corrected config with an appropriate `ent_coef` value and explain why entropy regularisation is especially important for sparse-reward tasks.

<details>
<summary>🔑 Reveal answer — Config B diagnosis</summary>

**The problem:** `ent_coef=0.0` disables entropy regularisation entirely. In a sparse-reward environment the agent receives no reward signal during early training — every rollout returns zero for every transition. Without a gradient signal to shape the policy, the network's parameter noise causes it to drift toward a deterministic distribution based on random initialisation. With no entropy term penalising low-entropy policies, this collapse happens quickly and irreversibly: the agent commits to a narrow set of actions before it has ever found the goal, and thereafter never explores the state space broadly enough to discover the `+1.0` reward.

**The fix:** Set `ent_coef` to 0.01–0.05. Higher values (0.05) are appropriate early in training for sparse tasks because the entropy bonus keeps the policy exploring even when all observed returns are zero.

```python
model_B_fixed = PPO(
    "MlpPolicy",
    make_vec_env(ENV_NAME, n_envs=8),
    n_steps=1024,
    batch_size=512,
    n_epochs=10,
    learning_rate=3e-4,
    clip_range=0.2,
    ent_coef=0.01,     # non-zero: prevents premature policy collapse before first success
    verbose=1,
)
```

</details>

In [ ]:
# Fix for Config B

# model_B_fixed = PPO(
#     "MlpPolicy",
#     make_vec_env(ENV_NAME, n_envs=8),
#     n_steps=1024,
#     batch_size=512,
#     n_epochs=10,
#     learning_rate=3e-4,
#     clip_range=0.2,
#     ent_coef=???,        # your value here — justify in a comment
#     verbose=1,
# )


### Revised prompt for Config B

**Original prompt:**
> "Generate a PPO config for a sparse-reward pick-and-place robotic task."

**Your revised prompt:**

*(Write here. Specify what you want the entropy coefficient to do and why.)*

<details>
<summary>🔑 Model prompt — Config B</summary>

**Example strong prompt:**

> Write a Stable-Baselines3 PPO config for a sparse-reward pick-and-place robotic task. The reward is +1.0 only on task success; the agent may go thousands of timesteps without any reward signal early in training. Set `ent_coef` to a non-zero value (0.01–0.05) to maintain exploration entropy throughout training — explain in a comment why entropy regularisation is especially important for sparse-reward environments. Do not set `ent_coef=0.0`.

**Why it's strong:** It identifies the failure mode (premature policy collapse without reward), states the required constraint (`ent_coef > 0`) as a direct instruction, and asks for an inline justification comment so the generated code is self-explanatory.

</details>

---

## Config C

```python
model_C = PPO(
    "MlpPolicy",
    make_vec_env(ENV_NAME, n_envs=8),
    n_steps=1024,
    batch_size=512,
    n_epochs=10,
    learning_rate=3e-4,
    clip_range=0.2,
    ent_coef=0.01,
    verbose=1,
)
model_C.learn(total_timesteps=5_000_000)
```

### Diagnosis

**Is Config C acceptable as a starting point for this task, or does it have a problem?**

*(Write your answer here. If it is acceptable, explain why each hyperparameter is reasonable for this scenario. If it has a problem, identify it.)*

### Reflection

Config C uses `n_steps=1024` with `n_envs=8`, giving `8 × 1024 = 8 192` transitions per update. With episodes of 200–400 steps, roughly how many complete episodes will be in each rollout? Is the advantage estimate likely to contain non-zero reward signal? Justify your answer.

<details>
<summary>🔑 Reveal answer — Config C diagnosis</summary>

**Config C is an acceptable starting point.** Each hyperparameter is defensible for this task:

- `n_steps=1024`: 8 × 1024 = 8 192 transitions per rollout; with average episode length ~300 this yields ~27 complete episodes per rollout, giving the advantage estimator a realistic chance of seeing successful episodes once the agent starts learning.
- `batch_size=512`: standard mini-batch size relative to the 8 192-transition rollout; covers the rollout in ~16 mini-batches over 10 epochs.
- `n_epochs=10`, `clip_range=0.2`: PPO defaults, appropriate unless there is evidence of policy churn.
- `learning_rate=3e-4`: standard Adam rate for continuous-control tasks.
- `ent_coef=0.01`: non-zero entropy bonus — critical for sparse rewards; this is the minimum recommended value.

The config is not perfect (reward shaping or curriculum learning would help further) but it is a sound baseline that will learn rather than stall.

</details>

In [ ]:
# Compute: how many complete episodes per rollout?

n_envs   = 8
n_steps  = 1024
ep_len   = 300   # approximate average

transitions_per_rollout = n_envs * n_steps
approx_complete_episodes = transitions_per_rollout / ep_len

print(f"Transitions per rollout: {transitions_per_rollout}")
print(f"Approximate complete episodes per rollout: {approx_complete_episodes:.1f}")
print()
print("Will each rollout see at least some reward signal? Explain in the cell below.")


*(Write your explanation here.)*

<details>
<summary>🔑 Reveal answer — Config C reflection</summary>

**Calculation:** 8 envs × 1024 steps = 8 192 transitions per rollout. At an average episode length of 300 steps, each rollout spans roughly 8 192 / 300 ≈ **27 complete episodes**.

**Will rollouts contain reward signal?** Yes — with ~27 episodes per rollout and an environment that a learning agent can eventually solve, at least some episodes will return the +1.0 reward once the policy begins improving. Even early in training, 27 simultaneous attempts per rollout means the probability of at least one success is non-negligible once the agent has learned any useful partial behaviour. This contrasts sharply with Config A's ~1–2 partial episodes per rollout, which would almost never contain a terminal success.

</details>

---

## Final task: write a single robust prompt

Combine your findings into one prompt that would generate a correct, well-justified PPO config for this robotic arm task — without requiring the human to post-process the result.

**Your prompt:**

*(Write here. The prompt should specify: task type, observation/action space type, episode length range, reward structure, parallelism, training budget, and any constraints on hyperparameters that matter for this setting.)*

<details>
<summary>🔑 Model prompt — Final task</summary>

**Example strong prompt:**

> Write a complete Stable-Baselines3 PPO training script for a sparse-reward robotic pick-and-place task with the following properties:
>
> - **Environment:** `RoboticArm-v1` (substitute `Pendulum-v1` for a quick smoke test)
> - **Observation space:** 24-dimensional continuous vector (joint positions, velocities, object pose)
> - **Action space:** 6-dimensional continuous (joint torques); use `MlpPolicy`
> - **Reward structure:** sparse — +1.0 only on task completion, 0.0 otherwise
> - **Episode length:** 200–400 steps to success; 500-step hard cutoff
> - **Parallelism:** 8 parallel environments (`make_vec_env`)
> - **Training budget:** 5 000 000 timesteps
>
> Hyperparameter requirements:
> - `n_steps` must be ≥ 512 so each rollout spans at least one full episode per environment; justify your chosen value with a comment.
> - `ent_coef` must be > 0 (recommend 0.01–0.05); explain in a comment why entropy regularisation is critical for sparse rewards.
> - Other hyperparameters at PPO defaults unless you have a specific justification.
>
> Include a training loop that prints average reward every 50 000 timesteps and saves the final model.

**Why it's strong:** It provides every variable the model needs to reason about rollout coverage (`n_steps`), exploration (`ent_coef`), and policy architecture (`MlpPolicy` for continuous obs/action). The constraint clauses prevent the two most common failure modes seen in Configs A and B, and the comment requirement forces the generated code to be self-documenting.

</details>

## Summary

> **For each config, write one sentence on what was wrong (or right) and what the key question is to ask an AI tool before accepting a PPO config for a new environment.**

A.
B.
C.

**Key question to ask:**

<details>
<summary>🔑 Reveal summary answers</summary>

A. **Config A — n_steps too small:** `n_steps=64` produces rollouts shorter than a single episode, so the advantage estimator almost never sees a sparse reward and the policy cannot learn; always specify episode length and ensure n_steps ≥ max_episode_length.
B. **Config B — ent_coef=0.0:** Disabling entropy regularisation causes premature policy collapse before the agent finds its first success in a sparse environment; always set ent_coef > 0 (0.01–0.05) for sparse-reward tasks.
C. **Config C — acceptable baseline:** `n_steps=1024` covers ~27 episodes per rollout and `ent_coef=0.01` maintains exploration; this is a sound starting point for this task.

**Key question to ask before accepting a PPO config for a new environment:** *"What is the episode length and reward structure, and is n_steps long enough to capture complete episodes with non-zero returns?"*

</details>